# pvsamlab BESS Sandbox

Demonstrates all simulation modes after the Phase 3 BESS extension.

| Cell | Mode | PySAM module |
|------|------|--------------|
| 1 | Setup & imports | — |
| 2 | PV-only (`System`) | `Pvsamv1 en_batt=0` |
| 3 | PV + BESS (`PvBessSystem`) | `Pvsamv1 en_batt=1` |
| 4 | BESS-only (`StandaloneBessSystem`) | `StandAloneBattery` |
| 5 | Financial overlay (`compute_lcoe` + `compute_lcos`) | `SingleOwner` / `Cashloan` |
| 6 | Merchant price curve (`RevenueStack` + `price_signal` dispatch) | `SingleOwner` |
| 7 | 100 MW / 4-HR nameplate sizing loop | `StandAloneBattery` |

> **Weather data:** Uses cached NSRDB files for Scurry County, TX  
> `lat=33.0278, lon=-100.0814, year='2017'`  
> so no API calls are required to run this notebook.

In [ ]:
import json
import pprint

from pvsamlab import (
    # PV-only
    System,
    TrackingMode,
    # BESS
    Battery,
    BessDispatch,
    PvBessSystem,
    StandaloneBessSystem,
    # Financial
    Financial,
    RevenueStack,
    compute_lcoe,
    compute_lcos,
    compute_npv,
    compute_irr,
)

# ---------------------------------------------------------------------------
# Shared site parameters — reused in every cell below
# ---------------------------------------------------------------------------
SITE_LAT   = 33.0278
SITE_LON   = -100.0814
MET_YEAR   = '2017'           # cached NSRDB file — no download needed
TARGET_MW  = 100_000          # 100 MWac target plant size

print('pvsamlab imported successfully.')

---
## Cell 2 — PV-only (`System`)

Verifies that the existing PV-only path is unchanged after the BESS extension.

In [ ]:
# Build the PV-only system
pv_plant = System(
    lat=SITE_LAT,
    lon=SITE_LON,
    target_kwac=TARGET_MW,
    target_dcac=1.30,
    met_year=MET_YEAR,
    tracking_mode=TrackingMode.SAT,
)

print(f'System built:  {pv_plant.kwac:,.0f} kWac  |  DC/AC = {pv_plant.dc_ac_ratio:.3f}')

# Run simulation
pv_results = pv_plant.run()

print('\n--- PV-only results ---')
for k, v in pv_results.items():
    print(f'  {k:<50s} {v}')

---
## Cell 3 — PV + BESS (`PvBessSystem`)

Adds a 4-hour LFP battery co-located with the PV plant.  
Uses `Pvsamv1` with `en_batt=1` and **self-consumption** dispatch.

In [ ]:
# Battery: 400 MWh / 100 MW (4-hour duration), LFP, DC-coupled
batt = Battery(
    chemistry='LFP',
    energy_kwh=400_000,       # 400 MWh
    power_kw=100_000,         # 100 MW
    soc_min=10.0,
    soc_max=95.0,
    coupling='DC',
)

# Dispatch: automated self-consumption (charges from excess PV, discharges to load)
disp = BessDispatch(strategy='self_consumption')

# Load profile: flat 80 MW demand — gives self-consumption a discharge signal.
# Without a load profile the battery has no demand to discharge against.
load_profile_kw = [80_000.0] * 8760

# Build PV+BESS system — inherits all PV fields from System
pvbess_plant = PvBessSystem(
    lat=SITE_LAT,
    lon=SITE_LON,
    target_kwac=TARGET_MW,
    target_dcac=1.30,
    met_year=MET_YEAR,
    tracking_mode=TrackingMode.SAT,
    battery=batt,
    dispatch=disp,
    load_profile=load_profile_kw,
)

print(f'PvBessSystem built:  {pvbess_plant.kwac:,.0f} kWac  |  '
      f'Battery: {batt.energy_kwh:,.0f} kWh / {batt.power_kw:,.0f} kW')

# Run simulation
pvbess_results = pvbess_plant.run()

print('\n--- PV+BESS results ---')
for k, v in pvbess_results.items():
    print(f'  {k:<50s} {v}')


---
## Cell 4 — BESS-only (`StandaloneBessSystem`)

No PV.  Simulates a standalone battery against a flat 8760-hour load profile.  
Uses `PySAM.StandAloneBattery`.  Downloads ambient temperature from the same
cached NSRDB file.

In [ ]:
import numpy as np

# Flat load profile: 50 MW constant for all 8760 hours
flat_load_kw = [50_000.0] * 8760

# Battery: 200 MWh / 50 MW (4-hour), LFP, AC-coupled (standalone requires AC)
sa_batt = Battery(
    chemistry='LFP',
    energy_kwh=200_000,
    power_kw=50_000,
    soc_min=10.0,
    soc_max=95.0,
)

# Dispatch: self-consumption — the Battery standalone module's manual dispatch
# (batt_dispatch_choice=0) is non-functional in PySAM 6 standalone mode;
# self-consumption (choice=3) correctly cycles the battery against the load profile.
sa_disp = BessDispatch(strategy='self_consumption')

standalone = StandaloneBessSystem(
    battery=sa_batt,
    dispatch=sa_disp,
    load_profile=flat_load_kw,
    lat=SITE_LAT,
    lon=SITE_LON,
    weather_year=MET_YEAR,
)

print(f'StandaloneBessSystem built:  '
      f'{sa_batt.energy_kwh:,.0f} kWh / {sa_batt.power_kw:,.0f} kW')

# Run simulation (downloads/uses cached ambient temperature)
sa_results = standalone.run()

print('\n--- BESS-only results ---')
for k, v in sa_results.items():
    print(f'  {k:<50s} {v}')


---
## Cell 5 — Financial overlay

Demonstrates `compute_lcoe` (PySAM SingleOwner / Cashloan) and `compute_lcos`
(pure-Python post-simulation) on top of the already-executed simulations.

For the LCOS calculation we construct a simple annual discharge projection:
year-1 discharge from the PV+BESS simulation, degraded by the battery
calendar degradation rate for each subsequent year.

In [ ]:
# ── Financial assumptions ──────────────────────────────────────────────────
fin = Financial(
    analysis_period=25,
    pv_capex_per_kwdc=700.0,      # $/kWdc
    opex_per_kwac_year=15.0,      # $/kWac/yr
    ppa_rate=30.0,                # $/MWh — gives PV-only IRR ~12-13 % (in target range)
    ppa_escalation=1.0,           # %/yr
    discount_rate=8.0,            # % real
    inflation_rate=2.5,
    debt_fraction=0.0,        # all-equity: avoids ITC-covers-all-equity artefact
    loan_rate=5.0,
    loan_term=18,
    federal_tax_rate=21.0,
    itc_rate=0.0,   # no ITC simplifies equity analysis; IRR driven by PPA/capex only
    depreciation_schedule='MACRS5',
)

# ── LCOE / IRR / NPV  (PySAM SingleOwner — kwac > 1000 kW) ────────────────
print('Running compute_lcoe on PV-only system...')
lcoe_pv = compute_lcoe(pv_plant, fin)

print('\n--- PV-only financials (SingleOwner) ---')
for k, v in lcoe_pv.items():
    print(f'  {k:<40s} {v}')

# ── LCOE on PV+BESS system ─────────────────────────────────────────────────
print('\nRunning compute_lcoe on PV+BESS system...')
lcoe_pvbess = compute_lcoe(pvbess_plant, fin)

print('\n--- PV+BESS financials (SingleOwner) ---')
for k, v in lcoe_pvbess.items():
    print(f'  {k:<40s} {v}')

# ── LCOS (post-simulation Python) ─────────────────────────────────────────
# Project annual discharge for 25 years using calendar degradation
y1_discharge_kwh = pvbess_results['batt_annual_discharge_energy_kwh']
deg_rate = batt.calendar_degradation / 100.0   # fraction per year
annual_discharge = [
    y1_discharge_kwh * (1.0 - deg_rate) ** yr
    for yr in range(fin.analysis_period)
]

annual_bess_opex = batt.energy_kwh * batt.opex_per_kwh_year   # $/year
dr = fin.discount_rate / 100.0

lcos = compute_lcos(
    battery=batt,
    annual_discharge_kwh=annual_discharge,
    annual_opex=annual_bess_opex,
    replacement_events=[],   # no replacement assumed
    discount_rate=dr,
)

print(f'\n--- LCOS ---')
print(f'  Year-1 discharge           {y1_discharge_kwh:>15,.1f}  kWh')
print(f'  Annual BESS O&M            {annual_bess_opex:>15,.0f}  $/yr')
print(f'  LCOS                       {lcos:>15.4f}  $/kWh')

# ── Standalone NPV / IRR helpers ──────────────────────────────────────────
# Quick sanity-check using the pure-Python helpers with a toy cash-flow series
example_cashflows = [-1_000_000] + [120_000] * 25    # $1M capex, $120k/yr
ex_npv = compute_npv(example_cashflows, discount_rate=0.08)
ex_irr = compute_irr(example_cashflows)

print(f'\n--- Standalone NPV / IRR (toy example) ---')
print(f'  Cash flows: -$1M upfront, +$120k/yr × 25 years')
print(f'  NPV @ 8%   {ex_npv:>12,.2f}  $')
print(f'  IRR        {ex_irr * 100:>12.3f}  %')

---
## Cell 6 — Merchant price curve (`RevenueStack` + `price_signal` dispatch)

Two new features are demonstrated:

**Part A — Physical dispatch:**  
A 100 MW / 400 MWh standalone battery with `price_signal` dispatch (choice=4) and
grid-charging enabled (`can_gridcharge=[1]*6`).  A flat $35/MWh placeholder is used;
replace with an 8760-element CSV column to use real hourly prices.

**Part B — Financial overlay with `RevenueStack`:**  
`compute_lcoe` now accepts an optional `revenue_stack` argument.  When
`energy_arbitrage_prices` is provided, the 8760-hour array replaces the flat
`ppa_rate` in Singleowner.  Capacity and ancillary payments are added as a
post-simulation Python NPV bonus on top of the SAM result.  NPV is compared
against the flat PPA case from Cell 5.

In [ ]:
# ── Merchant price curve ───────────────────────────────────────────────────
# Flat $35/MWh placeholder; swap in a real hourly price CSV column to use
# actual market prices:
#   import pandas as pd
#   merchant_prices = pd.read_csv('prices.csv')['da_price_mwh'].tolist()[:8760]
MERCHANT_PRICE_MWH = 35.0
merchant_prices = [MERCHANT_PRICE_MWH] * 8760

# ── Part A: Standalone BESS — price_signal dispatch ───────────────────────
# can_gridcharge=[1]*6 allows the battery to charge from the grid when prices
# are low; batt_dispatch_auto_can_gridcharge is derived from this in battery.py.
merchant_disp = BessDispatch(
    strategy='price_signal',
    can_gridcharge=[1, 1, 1, 1, 1, 1],
)

merchant_batt = Battery(
    chemistry='LFP',
    energy_kwh=400_000,       # 400 MWh
    power_kw=100_000,         # 100 MW
    capex_per_kwh=250.0,
    capex_per_kw=150.0,
    opex_per_kwh_year=8.0,
)

# Front-of-meter: no behind-meter load (zeros); battery charges from / discharges to grid
standalone_merchant = StandaloneBessSystem(
    battery=merchant_batt,
    dispatch=merchant_disp,
    load_profile=[0.0] * 8760,
    lat=SITE_LAT,
    lon=SITE_LON,
    weather_year=MET_YEAR,
)

print('Running standalone BESS with price_signal dispatch...')
merchant_results = standalone_merchant.run()

print('\n--- Standalone BESS (price_signal dispatch, $35/MWh flat) ---')
for k, v in merchant_results.items():
    print(f'  {k:<50s} {v}')

# ── Part B: Financial overlay — RevenueStack on PV+BESS plant ─────────────
# energy_arbitrage_prices replaces the flat ppa_rate inside Singleowner.
# capacity and ancillary payments are added as a post-simulation Python NPV bonus.
rev = RevenueStack(
    energy_arbitrage_prices=merchant_prices,
    capacity_payment_per_kw_year=80.0,    # $/kW-yr capacity market
    ancillary_services_per_kw_year=15.0,  # $/kW-yr ancillary services
)

print('\nRunning compute_lcoe on PV+BESS plant with merchant RevenueStack...')
lcoe_merchant = compute_lcoe(pvbess_plant, fin, revenue_stack=rev)

print('\n--- PV+BESS financials — merchant stack ---')
for k, v in lcoe_merchant.items():
    print(f'  {k:<40s} {v}')

# ── NPV comparison ─────────────────────────────────────────────────────────
delta_npv = lcoe_merchant['npv_usd'] - lcoe_pvbess['npv_usd']

print('\n--- NPV comparison: flat PPA vs merchant stack ---')
print(f'  Flat PPA ($30/MWh, no BESS revenue bonus)   ${lcoe_pvbess["npv_usd"]:>15,.0f}')
print(f'  Merchant ($35 arb + $80 cap + $15 anc/kW-yr) ${lcoe_merchant["npv_usd"]:>15,.0f}')
print(f'  Delta                                         ${delta_npv:>15,.0f}')
print(f'  Annual revenue bonus                          ${lcoe_merchant["annual_revenue_bonus_usd"]:>15,.0f}')

---
## Cell 7 — 100 MW / 4-HR nameplate sizing loop

Sweeps installed energy capacity from 400 MWh to 550 MWh (step 10 MWh) for a fixed
100 MW / 4-hour nameplate battery.  For each size the battery simulation is run in
single-year mode, `batt_capacity_percent` is projected over 25 years using the
calendar degradation rate, and the first year SOH falls below 100% is recorded.

The table shows:  `installed_MWh | years_at_nameplate | LCOS ($/kWh)`

The minimum installed capacity where `years_at_nameplate >= 10` is highlighted.

In [ ]:
import numpy as np

# ── Sizing loop parameters ─────────────────────────────────────────────────
POWER_KW       = 100_000          # fixed 100 MW
DURATION_H     = 4                # 4-hour nameplate
ANALYSIS_YEARS = 25               # project life

# Sweep: 400 MWh → 550 MWh, step 10 MWh
energy_sweep_kwh = list(range(400_000, 560_000, 10_000))

# Re-use Cell 4's self_consumption dispatch (consistent with prior cells)
sizing_disp = BessDispatch(strategy='self_consumption')
sizing_load = [50_000.0] * 8760   # flat 50 MW load (same as Cell 4)

# Financial parameters for LCOS (re-use Cell 5's discount_rate)
dr = fin.discount_rate / 100.0

# ── Run sweep ──────────────────────────────────────────────────────────────
rows = []

for energy_kwh in energy_sweep_kwh:
    sizing_batt = Battery(
        chemistry='LFP',
        energy_kwh=float(energy_kwh),
        power_kw=float(POWER_KW),
        capex_per_kwh=250.0,
        capex_per_kw=150.0,
        opex_per_kwh_year=8.0,
        calendar_degradation=2.0,   # %/year
    )

    sys_sizing = StandaloneBessSystem(
        battery=sizing_batt,
        dispatch=sizing_disp,
        load_profile=sizing_load,
        lat=SITE_LAT,
        lon=SITE_LON,
        weather_year=MET_YEAR,
    )
    results = sys_sizing.run()

    # Year-1 discharge from single-year simulation
    y1_discharge_kwh = results['batt_annual_discharge_energy_kwh']

    # Project SOH over analysis_period using linear calendar degradation.
    # Single-year mode returns batt_capacity_percent as a 1-element list [100.0];
    # we project it manually using the calendar_degradation rate.
    deg_pct_per_year = sizing_batt.calendar_degradation  # % per year
    soh_by_year = [max(0.0, 100.0 - deg_pct_per_year * yr) for yr in range(ANALYSIS_YEARS + 1)]

    # First year where SOH < 100%: year 0 = installation, year 1 = end of year 1
    years_at_nameplate = next(
        (yr - 1 for yr in range(1, ANALYSIS_YEARS + 1) if soh_by_year[yr] < 100.0),
        ANALYSIS_YEARS,
    )

    # Annual discharge projection: scale y1_discharge by SOH fraction each year
    annual_discharge = [
        y1_discharge_kwh * soh_by_year[yr + 1] / 100.0
        for yr in range(ANALYSIS_YEARS)
    ]

    annual_bess_opex = sizing_batt.energy_kwh * sizing_batt.opex_per_kwh_year
    lcos = compute_lcos(
        battery=sizing_batt,
        annual_discharge_kwh=annual_discharge,
        annual_opex=annual_bess_opex,
        replacement_events=[],
        discount_rate=dr,
    )

    rows.append({
        'installed_MWh': energy_kwh / 1000,
        'years_at_nameplate': years_at_nameplate,
        'lcos_usd_per_kwh': round(lcos, 4),
    })

    print(f'  {energy_kwh/1000:>5.0f} MWh  |  {years_at_nameplate:>3d} yrs  |  ${lcos:.4f}/kWh')

# ── Find minimum size with years_at_nameplate >= 10 ───────────────────────
qualifying = [r for r in rows if r['years_at_nameplate'] >= 10]
min_qualifying = qualifying[0] if qualifying else None

print('\n--- Sizing table summary ---')
print(f"{'installed_MWh':>15}  {'years_at_nameplate':>20}  {'LCOS ($/kWh)':>14}")
print('-' * 55)
for r in rows:
    marker = '  ← MIN (≥10 yr)' if min_qualifying and r['installed_MWh'] == min_qualifying['installed_MWh'] else ''
    print(f"  {r['installed_MWh']:>12.0f}  {r['years_at_nameplate']:>20d}  {r['lcos_usd_per_kwh']:>14.4f}{marker}")

if min_qualifying:
    print(f"\nMinimum installed capacity for ≥10 years at nameplate: "
          f"{min_qualifying['installed_MWh']:.0f} MWh  |  LCOS = ${min_qualifying['lcos_usd_per_kwh']:.4f}/kWh")
else:
    print('\nNo configuration achieves ≥10 years at nameplate within the sweep range.')